# TODO: Explain all that is happening here, the theory behind it

In [ ]:
import polyscope
import numpy as np
from pygel3d import hmesh

In [ ]:
mesh = hmesh.load("data/spot.obj")

In [3]:
allfaces = [[v for v in mesh.circulate_face(f)] for f in mesh.faces()]

polyscope.init()
polyscope.register_surface_mesh("My Mesh", vertices=np.array(mesh.positions()), faces=allfaces)

[polyscope] Backend: openGL3_glfw -- Loaded openGL version: 4.6 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2


In [4]:
def cotan_weight(p_i, p_j, p_k):
    # cotan of the angle to k in triangle (i,j,k)
    u = p_i - p_k
    v = p_j - p_k
    cot = np.dot(u, v) / np.linalg.norm(np.cross(u, v))
    return cot

In [5]:
def laplace_beltrami_matrix_f(mesh):
    # Returns M, L, M-1@L
    vertices = mesh.vertices()
    pos = mesh.positions()
    n_v = len(vertices)
    L = np.zeros(shape=(n_v, n_v))
    M_diag = np.zeros(n_v)
    for i in range(n_v):
        v_i = i
        
        faces_adj = mesh.circulate_vertex(v_i, mode="f")
        area_sum = 0
        for f in faces_adj:
            area_sum += mesh.area(f)
        M_diag[i] = area_sum / 3.0
        
        neighbors = mesh.circulate_vertex(v_i, mode="v")
        sum_weights = 0
        
        for v_j in neighbors:
            faces_ij = list(set(mesh.circulate_vertex(v_i, mode="f")) & 
                            set(mesh.circulate_vertex(v_j, mode="f")))
        
            w_ij = 0
            
            for face in faces_ij:
                f_verts = list(mesh.circulate_face(face, mode="v"))
                v_k = [v for v in f_verts if v != v_i and v != v_j][0]
                
                w_ij += cotan_weight(pos[v_i], pos[v_j], pos[v_k])
                
            w_ij = 0.5 * w_ij
            L[v_i, v_j] = -w_ij
            sum_weights += w_ij
        
        L[v_i, v_i] = sum_weights
        
    M_inv = np.diag(1.0 / M_diag)
    laplace_beltrami = M_inv @ L
            
    
    return laplace_beltrami, L, np.diag(M_diag)


In [6]:
laplace_beltrami_matrix, L, M = laplace_beltrami_matrix_f(mesh)

In [ ]:
from scipy import sparse

# I convert L and M to sparse matrices for efficient eigenvalue computation
L_sparse = sparse.csc_matrix(L)
M_sparse = sparse.csc_matrix(M)

k = 500  # number of eigenvalues/eigenvectors to compute

from scipy.sparse.linalg import eigsh
# We resolve L * phi = lambda * M * phi
# 'SM' is for 'Smallest Magnitude' eigenvalues, but since we want the smallest non-zero eigenvalues, we use 'sigma=0' to shift the spectrum
evals, evecs = eigsh(L_sparse, k=k, M=M_sparse, sigma=0, which='LM')

In [25]:
t = 0.01
hks_result = np.zeros((len(mesh.vertices()), k))
for j in range(k):
    hks_result[:, j] = np.exp(-evals[j] * t) * evecs[:, j]**2
    
hks = np.sum(hks_result, axis=1)
polyscope.get_surface_mesh("My Mesh").add_scalar_quantity("HKS", hks, defined_on="vertices")

In [27]:
polyscope.show()